# Bot Answerability Analysis — All Hotels, Last 3 Months
**Data:** `2026_04_17_messages.xls` — 5 hotels (БС74, МК16, Дубай, М73, О-44)  
**Pipeline:** Load → Filter (3 months, guest-only) → Threads → Classify (BOT / HUMAN)  
**Question:** Какие тредды может закрыть бот-консьерж без оператора?  

---
### Логика классификации
Бот МОЖЕТ ответить (BOT): вопросы про объект, wifi, правила, коды, расписание, процедуры  
Нужен человек (HUMAN): физические действия, ремонт, уборка, переезд, возвраты, медиафайлы, жалобы

In [1]:
# ── Cell 0: Setup ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import json, os, re
from datetime import datetime, timedelta
from tqdm.auto import tqdm
from openai import OpenAI

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH = "/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/data/2026_04_17_messages.xls"
OUT_PATH  = "/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/data/bot_answerability.parquet"

HOTEL_NAMES = {1: 'БС74', 2: 'МК16', 3: 'Дубай', 4: 'М73', 5: 'О-44'}
GAP_HOURS   = 4       # gap that starts a new thread
LOOKBACK_DAYS = 90    # last 3 months
BATCH_SIZE  = 10      # threads per GPT call
MODEL       = 'gpt-4o-mini'

client = OpenAI()  # uses OPENAI_API_KEY env var
print('OpenAI client ready')

OpenAI client ready


In [2]:
# ── Cell 1: Load & Filter ─────────────────────────────────────────────────────
df = pd.read_excel(DATA_PATH, engine='xlrd')
df['date_add']  = pd.to_datetime(df['date_add'])
df['is_admin']  = df['from_admin'].fillna(0).astype(int)
df['message']   = df['message'].astype(str).str.strip()
df['hotel_name']= df['hotel_id'].map(HOTEL_NAMES)
# Rename to match other notebooks
df = df.rename(columns={'booking_id': 'ID_booking'})

# Remove empty, test messages, HTML-only, pure file markers
SKIP_PATTERNS = r'^(nan|тест|test|бегу|file|\s*)$|^<div'
df = df[~df['message'].str.lower().str.match(SKIP_PATTERNS)].copy()
df = df[df['message'].str.len() >= 3].copy()

# Last 3 months
cutoff = df['date_add'].max() - timedelta(days=LOOKBACK_DAYS)
df = df[df['date_add'] >= cutoff].copy()

# Guest messages only (admin messages are responses, not what we classify)
guest = df[df['is_admin'] == 0].copy()

print(f"Период: {df['date_add'].min().date()} → {df['date_add'].max().date()}")
print(f"Отели:  {sorted(df['hotel_id'].unique())} → {[HOTEL_NAMES[h] for h in sorted(df['hotel_id'].unique())]}")
print(f"Броней: {df['ID_booking'].nunique()}")
print(f"Гостевых сообщений: {len(guest)}")
print()
print(guest.groupby('hotel_name').size().rename('guest_msgs'))

Период: 2026-01-17 → 2026-04-17
Отели:  [1, 2, 3] → ['БС74', 'МК16', 'Дубай']
Броней: 343
Гостевых сообщений: 1002

hotel_name
БС74     509
Дубай    397
МК16      96
Name: guest_msgs, dtype: int64


In [3]:
# ── Cell 2: Thread detection ──────────────────────────────────────────────────
# Run on all messages (including admin) so gaps are detected correctly
df_sorted = df.sort_values(['ID_booking', 'date_add']).reset_index(drop=True)
df_sorted['prev_time'] = df_sorted.groupby('ID_booking')['date_add'].shift(1)
df_sorted['gap_hrs']   = (df_sorted['date_add'] - df_sorted['prev_time']).dt.total_seconds() / 3600
df_sorted['new_thread']= df_sorted['prev_time'].isna() | (df_sorted['gap_hrs'] > GAP_HOURS)
df_sorted['thread_id'] = df_sorted.groupby('ID_booking')['new_thread'].cumsum()

# Build thread table — guest messages only, concatenated
guest_sorted = df_sorted[df_sorted['is_admin'] == 0].copy()

threads = (
    guest_sorted
    .groupby(['hotel_id', 'hotel_name', 'ID_booking', 'thread_id'])
    .agg(
        text         = ('message', lambda msgs: '\n---\n'.join(msgs)),
        n_guest_msgs = ('message', 'count'),
        thread_start = ('date_add', 'min'),
        thread_end   = ('date_add', 'max'),
    )
    .reset_index()
)

n = len(threads)
print(f"Тредов для классификации: {n}")
print(threads.groupby('hotel_name')['thread_id'].count().rename('threads'))

Тредов для классификации: 472
hotel_name
БС74     245
Дубай    166
МК16      61
Name: threads, dtype: int64


In [5]:
# ── Cell 3: GPT Classification — Bot Can Answer? ──────────────────────────────

SYSTEM_PROMPT = """
Ты — аналитик поддержки апарт-отеля. Оцени каждый тред гостевых сообщений:
может ли бот-консьерж ответить на него БЕЗ привлечения живого оператора?

БОТ может ответить, если гость:
- спрашивает информацию об объекте (wifi, коды, правила, удобства, расписание)
- запрашивает код ваучера / пин / инструкцию
- задаёт FAQ (парковка, заезд, выезд, прачечная)
- пишет нейтральное приветствие или благодарность

Нужен ЧЕЛОВЕК (оператор), если гость:
- просит что-то физически сделать (починить, убрать сейчас, переселить)
- жалуется на шум, соседей, качество обслуживания
- запрашивает возврат денег или оспаривает оплату
- сообщает о потерянных вещах
- хочет чтобы ему перезвонили
- отправляет файл / скрин / фото / видео
- сообщает о технической ошибке системы

Верни ТОЛЬКО JSON-массив объектов (без markdown, без пояснений):
[{"idx": <int>, "label": "BOT" или "HUMAN", "reason": "<1 фраза на рус. до 10 слов>", "confidence": <0.0-1.0>}]
""".strip()


def classify_batch(batch_df):
    items = []
    for i, row in enumerate(batch_df.itertuples()):
        preview = row.text[:400].replace('\n---\n', ' | ')
        items.append(f'[{i}] {preview}')

    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': '\n\n'.join(items)},
        ],
    )
    raw = resp.choices[0].message.content.strip()
    raw = re.sub(r'^```[a-z]*\n?|\n?```$', '', raw).strip()
    return json.loads(raw)


# ── Run classification — idx-aligned so length mismatches can't happen ────────
n = len(threads)
labels      = ['HUMAN'] * n   # default fallback
reasons     = [''] * n
confidences = [0.0] * n

for start in tqdm(range(0, n, BATCH_SIZE)):
    batch = threads.iloc[start: start + BATCH_SIZE]
    try:
        results = classify_batch(batch)
        for r in results:
            # r['idx'] is position within batch (0-based)
            global_idx = start + int(r['idx'])
            if 0 <= global_idx < n:          # guard against out-of-range
                labels[global_idx]      = r.get('label', 'HUMAN')
                reasons[global_idx]     = r.get('reason', '')
                confidences[global_idx] = float(r.get('confidence', 0.8))
    except Exception as e:
        print(f"  ⚠️  Batch {start}-{start+BATCH_SIZE} failed: {e}")

threads['label']      = labels
threads['reason']     = reasons
threads['confidence'] = confidences

print(f"\n{'='*40}")
print(threads['label'].value_counts())
pct = (threads['label'] == 'BOT').mean() * 100
print(f"\nБот может закрыть: {pct:.1f}% тредов")

  0%|          | 0/48 [00:00<?, ?it/s]


label
HUMAN    312
BOT      160
Name: count, dtype: int64

Бот может закрыть: 33.9% тредов


In [6]:
# ── Cell 4: Save & Summary ────────────────────────────────────────────────────
threads.to_parquet(OUT_PATH, index=False)
print(f"Сохранено: {OUT_PATH}")
print(f"Строк: {len(threads)}")
print()

# Per-hotel breakdown
summary = (
    threads
    .groupby(['hotel_name', 'label'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
)
if 'BOT' in summary.columns:
    summary['bot_pct'] = (summary['BOT'] / summary['total'] * 100).round(1)
print(summary)

Сохранено: /home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/data/bot_answerability.parquet
Строк: 472

label       BOT  HUMAN  total  bot_pct
hotel_name                            
БС74         79    166    245     32.2
Дубай        58    108    166     34.9
МК16         23     38     61     37.7


In [7]:
# ── Cell 5: Inspect BOT threads ───────────────────────────────────────────────
bot = threads[threads['label'] == 'BOT'].sort_values('confidence', ascending=False)
print(f"BOT тредов: {len(bot)}\n")
for _, row in bot.head(15).iterrows():
    preview = row['text'][:100].replace('\n---\n', ' | ')
    print(f"[{row['hotel_name']} / бронь {row['ID_booking']}]")
    print(f"  Reason: {row['reason']} (conf={row['confidence']:.2f})")
    print(f"  Текст:  {preview}")
    print()

BOT тредов: 160

[БС74 / бронь 1234]
  Reason: Нейтральное приветствие (conf=1.00)
  Текст:  Bonsoir

[БС74 / бронь 1338201]
  Reason: Запрос информации о продлении номера (conf=1.00)
  Текст:  здравствуйте! продлили номер на сутки, Бронь 20260413-16579-427589539

[Дубай / бронь 327636]
  Reason: Нейтральное приветствие (conf=1.00)
  Текст:  Hello

[МК16 / бронь 21704177]
  Reason: Запрос информации о прачечной (conf=1.00)
  Текст:  Здравствуйте, прачечная не заработала ?

[МК16 / бронь 21697167]
  Reason: Информация о комплиментарном ужине (conf=1.00)
  Текст:  Ужин комплиментарный | Информация у администратора ресепшн

[МК16 / бронь 2778]
  Reason: Нейтральное сообщение (conf=1.00)
  Текст:  Вот | Тест цитат

[МК16 / бронь 2778]
  Reason: Нейтральное сообщение (conf=1.00)
  Текст:  Тест1

[БС74 / бронь 11111]
  Reason: Запрос ссылки на апартаменты (conf=1.00)
  Текст:  Здравствуйте | Отправьте пожалуйста ссылку на суточные апартаменты из приложения

[МК16 / бронь 2778]
  Reason: Нейт

In [8]:
# ── Cell 6: Inspect HUMAN threads (spot-check) ────────────────────────────────
human = threads[threads['label'] == 'HUMAN'].sort_values('confidence', ascending=False)
print(f"HUMAN тредов: {len(human)}\n")
for _, row in human.head(10).iterrows():
    preview = row['text'][:100].replace('\n---\n', ' | ')
    print(f"[{row['hotel_name']} / бронь {row['ID_booking']}]")
    print(f"  Reason: {row['reason']} (conf={row['confidence']:.2f})")
    print(f"  Текст:  {preview}")
    print()

HUMAN тредов: 312

[МК16 / бронь 21691671]
  Reason: Просит что-то физически сделать (conf=1.00)
  Текст:  Здравствуйте, можете пожалуйста предоставить вторую пару полотенец. У нас только одна 🥺

[Дубай / бронь 333166]
  Reason: Проблемы с температурой и дверью (conf=1.00)
  Текст:  Hello good afternoon | My room it’s to much cold | Has someone on living in the room 2? | Excu

[Дубай / бронь 332963]
  Reason: Гость жалуется на информацию о клининге (conf=1.00)
  Текст:  Я дико извиняюсь, но вы написали что для меня в первый месяц клининг доступен каждую неделю | Сказ

[Дубай / бронь 332963]
  Reason: Гость запрашивает клининг (conf=1.00)
  Текст:  Здравствуйте! | Нужен клининг в 105 номере | Это бесплатно же ведь? | Сегодня | Давайте | 

[Дубай / бронь 332963]
  Reason: Гость жалуется на Wi-Fi (conf=1.00)
  Текст:  Здравствуйте! В 105 номере очень плохо работает вай фай | Решите вопрос пожалуйста

[Дубай / бронь 332963]
  Reason: Гость запрашивает возврат депозита (conf=1.00)
  Текст: 